# DNS-over-TLS Forwarding Lab

Functional test matrix for DNS servers in **forwarding configurations**: each server
under test listens on 53 (Do53) and 853 (DoT) and forwards the `lab.test` zone to a
framework-owned upstream pair:

| upstream | listens | `transport-marker.lab.test TXT` |
|---|---|---|
| `lab-auth-do53` | 53 only | `"do53"` |
| `lab-auth-dot` | 853/TLS only | `"dot"` |

Because the DoT upstream has **no** Do53 listener and each upstream serves a different
marker value, the answer content proves which transport the forwarder really used —
no packet capture needed. Servers/profiles are defined under
`dnslab/servers/<name>/` (see `dnslab.registry()`); capability flags make
unsupported combinations report **SKIP**, never a misleading FAIL.


In [ ]:
import dnslab
import pandas as pd
from rich import print as rprint

# Servers to test this run: (server, profile). Profiles ending in -dot forward
# upstream over TLS; -do53 is the plaintext-forwarding baseline/negative control.
SERVERS = [
    # ('unbound',       'forwarder-dot'),
    # ('bind',          'forwarder-do53'),
    # ('knot-resolver', 'forwarder-dot'),
    # ('knot',          'auth-dot'),
    # ('nsd',           'auth-dot'),
    # ('windows-dns',   'forwarder-do53'),   # tier 2: launches an EC2 instance ($)
]

registry = dnslab.registry()
pd.DataFrame([
    {'server': s.name, 'tier': s.tier, 'provider': s.provider,
     'roles': '/'.join(sorted(s.capabilities.roles)),
     'dot_listener': s.capabilities.dot_listener,
     'dot_upstream_fwd': s.capabilities.dot_upstream_forwarding,
     'profiles': ', '.join(sorted(s.profiles)), 'notes': s.capabilities.notes}
    for s in registry.values()
]).set_index('server')


## 1. Start the lab upstreams

Always first — forwarder configs are rendered with the upstreams' live addresses.


In [ ]:
for inst in dnslab.start('lab-auth'):
    print(f'{inst.name:16s} {inst.ip}  {inst.status}')
dnslab.status()


## 2. Sanity: query the upstreams directly


In [ ]:
for r in [c for t in dnslab.targets('lab-auth-do53', 'lab-auth-dot')
          for c in dnslab.checks.run_checks(t)]:
    print(f'{r.target:16s} {r.check:14s} {r.status:5s} {r.detail}')


## 3. Start the servers under test


In [ ]:
for name, profile in SERVERS:
    for inst in dnslab.start(name, profile=profile):
        print(f'{inst.name:16s} {inst.profile:16s} {inst.ip}  {inst.status}')
dnslab.status()


## 4. Functional test matrix

Columns: plain query on 53, TLS query on 853, cert chain/SAN validation,
forwarding resolution of `lab.test`, and the DoT-upstream marker proof.


In [ ]:
under_test = [t for t in dnslab.targets() if not t.name.startswith('lab-auth')]
matrix = dnslab.checks.run_matrix(under_test or dnslab.targets())
matrix.style.map(lambda v: {'PASS': 'background-color:#1a7f37;color:white',
                            'FAIL': 'background-color:#cf222e;color:white',
                            'SKIP': 'color:#888'}.get(v, ''))


In [ ]:
# detail behind every non-PASS cell
dnslab.checks.explain(matrix)


## 5. Debugging helpers


In [ ]:
# print(dnslab.logs('unbound', tail=50))
# print(open(dnslab.ca_file()).read())      # the lab CA notebooks/kdig validate against
# !kdig +tls-ca={dnslab.ca_file()} @lab-auth-dot.dnslab.test transport-marker.lab.test TXT


## 6. Teardown


In [ ]:
# dnslab.stop(all=True)     # stop every running instance (docker + EC2)
# dnslab.nuke()             # ...and delete tagged EC2 instances/security groups
dnslab.status()
